# Chapitre 2 — pandas : les fondamentaux

**Cours : Analyse et traitement des données** · Auteur : *Issa Gueye*

## Objectifs

- Manipuler les deux structures de base : **`Series`** et **`DataFrame`**.
- Sélectionner des lignes et colonnes avec **`loc`**, **`iloc`** et des **masques booléens**.
- Créer, modifier, trier, compter.
- Connaître les **nouveautés de pandas 3.0** (Copy-on-Write, type `str`, `pd.col`).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = next(p for p in [Path("data/brutes"), Path("../data/brutes"), Path("../../data/brutes")] if p.exists())
pd.set_option("display.max_columns", 20)
print("pandas", pd.__version__)

pandas 3.0.5


## 1. `Series` : un tableau 1D avec un index

In [2]:
prix = pd.Series([12500, 6500, 700], index=["Riz 25kg", "Huile 5L", "Sucre 1kg"], name="prix_fcfa")
print(prix)
print("Prix de l'huile :", prix["Huile 5L"])
print(prix * 1.18)   # vectorisé, comme NumPy, l'index est conservé

Riz 25kg     12500
Huile 5L      6500
Sucre 1kg      700
Name: prix_fcfa, dtype: int64
Prix de l'huile : 6500
Riz 25kg     14750.0
Huile 5L      7670.0
Sucre 1kg      826.0
Name: prix_fcfa, dtype: float64


L'**index** aligne automatiquement les données lors des opérations :

In [3]:
stock = pd.Series({"Sucre 1kg": 40, "Riz 25kg": 10, "Pain": 120})
print(prix * stock)   # alignement par étiquette ; NaN là où l'une des deux manque

Huile 5L          NaN
Pain              NaN
Riz 25kg     125000.0
Sucre 1kg     28000.0
dtype: float64


## 2. `DataFrame` : un tableau 2D de colonnes typées

Chaque colonne est une `Series` ; toutes partagent le même index de lignes.

In [4]:
df = pd.DataFrame({
    "region": ["Dakar", "Thiès", "Kaolack", "Dakar", "Ziguinchor"],
    "milieu": ["Urbain", "Urbain", "Rural", "Urbain", "Rural"],
    "taille_menage": [6, 9, 12, 4, 8],
    "revenu": [350_000, 210_000, 95_000, 520_000, 130_000],
})
df

,region,milieu,taille_menage,revenu
0,Dakar,Urbain,6,350000
1,Thiès,Urbain,9,210000
2,Kaolack,Rural,12,95000
3,Dakar,Urbain,4,520000
4,Ziguinchor,Rural,8,130000


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   region         5 non-null      str  
 1   milieu         5 non-null      str  
 2   taille_menage  5 non-null      int64
 3   revenu         5 non-null      int64
dtypes: int64(2), str(2)
memory usage: 353.0 bytes


> 🆕 **pandas 3.0** : les colonnes de texte ont maintenant le type dédié **`str`** (au lieu de `object`).
> C'est plus rapide (surtout avec `pyarrow` installé) et plus sûr : on ne peut plus mettre
> accidentellement un entier dans une colonne de texte.

In [6]:
print(df.dtypes)
print(df.shape, len(df), list(df.columns))

region             str
milieu             str
taille_menage    int64
revenu           int64
dtype: object
(5, 4) 5 ['region', 'milieu', 'taille_menage', 'revenu']


## 3. Lire un vrai fichier

In [7]:
menages = pd.read_csv(DATA / "enquete_menages.csv")
menages.head()

,id_menage,region,milieu,taille_menage,age_chef,sexe_chef,education_chef,revenu_mensuel_fcfa,depenses_mensuelles_fcfa,acces_electricite,acces_internet,utilise_mobile_money
0,M10000,Diourbel,Rural,6,36.0,F,Aucun,59000.0,42000.0,Oui,Non,Oui
1,M10001,Thiès,Urbain,6,78.0,F,Primaire,NaN,144000.0,Oui,Non,Non
2,M10002,Dakar,Urbain,8,44.0,F,Aucun,525000.0,489000.0,Oui,Non,Oui
3,M10003,Dakar,Urbain,5,56.0,F,Primaire,155000.0,124000.0,Oui,Non,Non
4,M10004,Dakar,Urbain,3,44.0,H,Supérieur,435000.0,343000.0,Oui,Oui,Oui


In [8]:
menages.describe()                      # résumé des colonnes numériques

,taille_menage,age_chef,revenu_mensuel_fcfa,depenses_mensuelles_fcfa
count,3000.000000,2944.000000,2.646000e+03,3.000000e+03
mean,8.773333,48.205163,2.800125e+05,2.369500e+05
std,9.974390,11.957537,2.229455e+05,1.893135e+05
min,1.000000,20.000000,1.800000e+04,1.500000e+04
25%,6.000000,40.000000,1.372500e+05,1.140000e+05
50%,8.000000,48.000000,2.140000e+05,1.815000e+05
75%,10.000000,56.000000,3.490000e+05,3.020000e+05
max,99.000000,90.000000,2.790000e+06,2.393000e+06


In [9]:
menages.describe(include="str")         # résumé des colonnes texte (pandas 3 : "str")

,id_menage,region,milieu,sexe_chef,education_chef,acces_electricite,acces_internet,utilise_mobile_money
count,3000,3000,3000,3000,3000,3000,3000,3000
unique,3000,8,2,2,5,2,2,2
top,M10000,Dakar,Urbain,H,Aucun,Oui,Non,Oui
freq,1,876,1702,2234,823,2252,1656,1935


## 4. Sélectionner des colonnes

In [10]:
menages["region"].head(3)                       # une colonne -> Series
menages[["region", "revenu_mensuel_fcfa"]].head(3)  # liste de colonnes -> DataFrame

,region,revenu_mensuel_fcfa
0,Diourbel,59000.0
1,Thiès,NaN
2,Dakar,525000.0


## 5. Sélectionner des lignes : `loc` vs `iloc`

| Accesseur | Sélectionne par… | Borne de fin | Exemple |
|---|---|---|---|
| `.loc[lignes, colonnes]` | **étiquettes** (noms, booléens) | **incluse** | `df.loc[0:4, "region"]` |
| `.iloc[lignes, colonnes]` | **positions** entières | **exclue** | `df.iloc[0:4, 1]` |

In [11]:
print(menages.loc[0:2, ["region", "milieu"]])   # lignes d'étiquette 0, 1, 2
print(menages.iloc[0:2, [1, 2]])                # positions 0 et 1

     region  milieu
0  Diourbel   Rural
1     Thiès  Urbain
2     Dakar  Urbain
     region  milieu
0  Diourbel   Rural
1     Thiès  Urbain


### Filtrer avec des conditions (masques booléens)

In [12]:
urbain_dakar = menages[(menages["region"] == "Dakar") & (menages["milieu"] == "Urbain")]
print(len(urbain_dakar))

sud = menages[menages["region"].isin(["Ziguinchor", "Kolda", "Sédhiou"])]
grands = menages.query("taille_menage >= 15 and taille_menage != 99")   # syntaxe alternative lisible
print(len(sud), len(grands))

834
260 63


## 6. Créer et modifier des colonnes

In [13]:
menages["epargne_fcfa"] = menages["revenu_mensuel_fcfa"] - menages["depenses_mensuelles_fcfa"]
menages["revenu_par_tete"] = menages["revenu_mensuel_fcfa"] / menages["taille_menage"]
menages["est_urbain"] = menages["milieu"] == "Urbain"
menages[["revenu_mensuel_fcfa", "depenses_mensuelles_fcfa", "epargne_fcfa", "revenu_par_tete"]].head()

,revenu_mensuel_fcfa,depenses_mensuelles_fcfa,epargne_fcfa,revenu_par_tete
0,59000.0,42000.0,17000.0,9833.333333
1,NaN,144000.0,NaN,NaN
2,525000.0,489000.0,36000.0,65625.000000
3,155000.0,124000.0,31000.0,31000.000000
4,435000.0,343000.0,92000.0,145000.000000


`assign` permet d'enchaîner les transformations (style « *method chaining* ») sans modifier l'original.

> 🆕 **pandas 3.0** introduit `pd.col("nom")` pour faire référence à une colonne dans `assign`
> sans écrire de `lambda`.

In [14]:
resume = (
    menages
    .assign(revenu_k=pd.col("revenu_mensuel_fcfa") / 1000,
            part_depenses=pd.col("depenses_mensuelles_fcfa") / pd.col("revenu_mensuel_fcfa"))
    .loc[:, ["region", "revenu_k", "part_depenses"]]
    .head()
)
resume

,region,revenu_k,part_depenses
0,Diourbel,59.0,0.711864
1,Thiès,NaN,NaN
2,Dakar,525.0,0.931429
3,Dakar,155.0,0.800000
4,Dakar,435.0,0.788506


Équivalent compatible avec pandas 2 : `.assign(revenu_k=lambda d: d["revenu_mensuel_fcfa"] / 1000)`.

### Modifier des valeurs sous condition : toujours avec `.loc`

In [15]:
menages.loc[menages["taille_menage"] == 99, "taille_menage"] = np.nan   # 99 = code « ne sait pas »
print(menages["taille_menage"].max())

22.0


> 🆕 **pandas 3.0 — Copy-on-Write (CoW)** : toute sélection se comporte comme une **copie**.
> L'« affectation en chaîne » ne modifie **jamais** le DataFrame d'origine :
>
> ```python
> df["col"][df["x"] > 5] = 0          # ❌ n'a aucun effet en pandas 3 (avertissement)
> df.loc[df["x"] > 5, "col"] = 0      # ✅ la bonne façon
> ```
>
> Le célèbre `SettingWithCopyWarning` disparaît : la règle est désormais simple et prévisible.

In [16]:
sous_df = menages[menages["region"] == "Dakar"]
sous_df.loc[:, "revenu_mensuel_fcfa"] = 0          # modifie la copie seulement…
print(menages.loc[menages["region"] == "Dakar", "revenu_mensuel_fcfa"].head(3))  # …l'original est intact

2    525000.0
3    155000.0
4    435000.0
Name: revenu_mensuel_fcfa, dtype: float64


## 7. Trier, compter, valeurs uniques

In [17]:
menages.sort_values("revenu_mensuel_fcfa", ascending=False).head(3)[["region", "revenu_mensuel_fcfa"]]

,region,revenu_mensuel_fcfa
2106,Kaolack,2790000.0
1855,Diourbel,2220000.0
2985,Dakar,1966000.0


In [18]:
print(menages["region"].value_counts())
print(menages["education_chef"].value_counts(normalize=True).round(3))   # proportions
print(menages["region"].nunique(), menages["region"].unique()[:4])

region
Dakar          876
Thiès          434
Diourbel       387
Saint-Louis    319
Kaolack        308
Ziguinchor     260
Louga          227
Tambacounda    189
Name: count, dtype: int64
education_chef
Aucun         0.274
Primaire      0.245
Secondaire    0.209
Coranique     0.167
Supérieur     0.104
Name: proportion, dtype: float64
8 <ArrowStringArray>
['Diourbel', 'Thiès', 'Dakar', 'Louga']
Length: 4, dtype: str


## 8. Statistiques par colonne et par groupe (aperçu)

In [19]:
print(menages["revenu_mensuel_fcfa"].agg(["mean", "median", "std", "min", "max"]).round(0))

mean       280012.0
median     214000.0
std        222945.0
min         18000.0
max       2790000.0
Name: revenu_mensuel_fcfa, dtype: float64


In [20]:
menages.groupby("milieu")["revenu_mensuel_fcfa"].median()

milieu
Rural     161000.0
Urbain    283500.0
Name: revenu_mensuel_fcfa, dtype: float64

Le chapitre 5 détaille `groupby`, les jointures et les tableaux croisés.

## 9. Types de données (dtypes) à connaître

| dtype pandas | Usage | Valeur manquante |
|---|---|---|
| `int64` | entiers (sans manquant) | — |
| `Int64` (majuscule) | entiers **nullables** | `<NA>` |
| `float64` | réels | `NaN` |
| `bool` / `boolean` | vrai/faux / nullable | `<NA>` |
| `str` (pandas 3) | texte | `NaN` |
| `category` | peu de modalités répétées (économise la mémoire, ordonnable) | `NaN` |
| `datetime64[us]` | dates/heures (pandas 3 : résolution micro-seconde par défaut) | `NaT` |

In [21]:
menages["education_chef"] = pd.Categorical(
    menages["education_chef"],
    categories=["Aucun", "Coranique", "Primaire", "Secondaire", "Supérieur"],
    ordered=True,
)
print(menages["education_chef"].dtype)
print((menages["education_chef"] >= "Secondaire").mean().round(3), "des chefs ont au moins le secondaire")
print(f"Mémoire : {menages.memory_usage(deep=True).sum() / 1e6:.2f} Mo")

category
0.313 des chefs ont au moins le secondaire
Mémoire : 0.40 Mo


## À retenir

- `Series` = colonne indexée ; `DataFrame` = table de colonnes partageant un index.
- **`loc` = étiquettes (fin incluse)**, **`iloc` = positions (fin exclue)**.
- Filtrer : `df[(cond1) & (cond2)]`, `isin`, `query`.
- Modifier sous condition : **toujours** `df.loc[masque, "col"] = valeur`.
- pandas 3.0 : type `str` par défaut, **Copy-on-Write** systématique, `pd.col()` dans `assign`.

**Références** : McKinney W. (2022), *Python for Data Analysis*, 3ᵉ éd., O'Reilly (gratuit en ligne :
<https://wesmckinney.com/book/>) ; notes de version pandas 3.0 :
<https://pandas.pydata.org/docs/whatsnew/v3.0.0.html>.

➡️ **Chapitre suivant : importer, exporter et auditer la qualité des données.**